# 3.2 Scaling laws: what more data buys, and what it asks of the model

At the end of this notebook you can:

- **read a scaling law**: the loss falls as a power of the data, the parameters or the
  compute, a straight line on log-log axes whose slope says what 10× more buys, and only
  inside the range that was measured;
- **say why data and parameters grow together**: on little data, extra parameters buy
  almost nothing; with the model held fixed, extra data buys less and less;
- **measure such a curve yourself** and draw a conclusion from it that goes beyond the top
  score.

**Why this matters.** You have a problem and a few thousand labelled examples. You read a
paper, take its architecture, its size and its training length, and train it on your data.
The paper trained on a thousand times more. The model does not learn, and machine learning
looks hard. The scaling laws say why: what a model of a
given size reaches depends on how much data it gets, and the right size, the right number
of steps and the right amount of data are read off a curve of your own problem. This
notebook measures such curves.

The two slides of lesson 3 on scaling laws showed the ratios of Kaplan et al. (2020) and
Hoffmann et al. (2022). Here you first measure your own curve, then read theirs next to it.

## 3.2.1 The data: Food-101 through a frozen encoder

Food-101 holds photos of 101 dishes, from apple pie to waffles. The cache on the hub holds
one 1024-dim vector per photo from `dinov2-large`, a pretrained image encoder as in
lesson 1, and the label of the dish. The encoder is **frozen**: its weights stay as they
are, and only a head on top of the vectors is trained.

The cache has two splits:

- **train**, 750 photos per class. Food-101's authors did not clean these labels by hand,
  so some photos carry the wrong dish. `food_scaling.py` (next to this notebook) keeps
  50 photos per class of it apart as the **validation** set, and takes the training images
  from the other 700;
- **test**, 250 photos per class, labelled by hand.

As in lesson 2, validation chooses and test judges. Every choice here (the step at which a
run is read) is made on validation; test has the hand-checked labels, so the curves are
read on test.

A training set of 10 images per class takes the first 10 of every class in one fixed order,
so a smaller set is always part of a larger one. The next cell downloads the cache once and
loads it.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from food_scaling import CLASSES, PER_CLASS, load, train
from notebooktester import param

food = load()
print(f"train      {tuple(food.X.shape)}   (of which 50 per class are validation)")
print(f"validation {tuple(food.X_valid.shape)}")
print(f"test       {tuple(food.X_held.shape)}")
print(
    f"{CLASSES} classes; test photos per class: {torch.bincount(food.y_held).unique().tolist()}"
)

## 3.2.2 Your own data axis

**Question.** As a head gets more training images, how far does its test loss fall, and
how many optimizer steps does it need to get there?

**Setup.** Lesson 1's linear head, `Linear(1024, 101)`: 1024 · 101 + 101 = 103,525
parameters. It is trained on 10, 30, 70, 200 and 700 images per class: 1,010 to 70,700
images, a factor of 70.

One **optimizer step** is one batch of 128 images: a forward pass, a backward pass and one
update of the weights. Every run trains well past its lowest validation loss; the step of
that lowest point is chosen, and the test loss is read at that step. One seed per size.

**Expect.** Write down before you run the cell: with more images, does the chosen step
come earlier or later, in optimizer steps? Is the test loss there lower or higher?

In [ ]:
SIZES = param(PER_CLASS, test=PER_CLASS[:3])

runs = {}
for pc in SIZES:
    X, y = food.subset(pc)
    runs[pc] = train(0, X, y, food)  # width 0: the linear head

COLOURS = ["#E69F00", "#56B4E9", "#009E73", "#D55E00", "#0072B2", "#CC79A7"]
plt.figure(figsize=(8, 4.2))
for (pc, r), c in zip(runs.items(), COLOURS):
    curve = r["curve"]
    plt.plot(curve["step"], curve["held"], color=c, label=f"{r['n']:,} images")
    plt.plot(r["best_step"], r["held"], "o", color=c, ms=7)
plt.xscale("log")
plt.yscale("log")
plt.ylim(0.18, 1.5)
plt.xlabel("optimizer steps (batches of 128, log)")
plt.ylabel("test loss (cross-entropy, log)")
plt.title("With more images, the chosen step comes later and its test loss lies lower")
plt.legend(frameon=False)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

Every curve starts at its first evaluation, after half a pass over its images, so a larger
set starts further to the right. Both axes are logarithmic: equal distances are equal
factors. The dot is the step chosen on validation; the test loss is read there. The next
cell prints the numbers at the dot, and then what happens when the smallest set is trained
for as many epochs as the largest one needed.

In [ ]:
print(
    f"{'images':>7s} {'step at the dot':>16s} {'epochs':>7s} {'test loss':>10s} {'test accuracy':>14s}"
)
for r in runs.values():
    epochs = r["best_step"] / (r["n"] / 128)
    print(
        f"{r['n']:7,d} {r['best_step']:16d} {epochs:7.1f} {r['held']:10.3f} {r['held_acc']:14.3f}"
    )

# the epochs of the largest run, used on the smallest set
small, large = runs[SIZES[0]], runs[SIZES[-1]]
epochs = large["best_step"] / (large["n"] / 128)
steps = epochs * small["n"] / 128
at = int(np.argmin(np.abs(np.array(small["curve"]["step"]) - steps)))
print(
    f"\n{epochs:.1f} epochs on {small['n']:,} images are {steps:.0f} steps: "
    f"test loss {small['curve']['held'][at]:.3f}, against {small['held']:.3f} at its own dot"
)

**Reading.** From 1,010 to 70,700 images the test loss at the dot falls from 0.482 to
0.225, and the step of the dot moves from 180 to 1,104: more data asks for more steps
before the head has learnt what it can. Counted in passes over the data (epochs) it goes the
other way, from 23 to 2: an epoch over more images holds more steps. Copy the 2 epochs of
the largest run onto the smallest set and the head stops after 16 steps, at a test loss of
1.1 instead of 0.48. A training length taken from someone else's data says little
about yours; your own validation curve says when to stop.

## 3.2.3 A straight line on log-log axes

A **power law** says the loss is a constant times the data to a negative power:
$L(D) = a \cdot D^{-\alpha}$, where $D$ is here the number of training images. Take the log of both sides and it is a straight line,
$\log L = \log a - \alpha \log D$, with slope $-\alpha$. So 10× more data multiplies the
loss by $10^{-\alpha}$: that factor, read off the slope, is what the law tells you.

The next cell draws the test loss at the dot of every run against the number of images,
both on log axes. It fits a straight line through the three smallest sizes only, and then
checks the line on the two larger sizes it has not seen.

**Expect.** Write down: will the two larger sizes lie on the line, above it or below it?

In [ ]:
D = np.array([r["n"] for r in runs.values()], dtype=float)
L = np.array([r["held"] for r in runs.values()])
slope, intercept = np.polyfit(np.log10(D[:3]), np.log10(L[:3]), 1)
line = lambda d: 10 ** (intercept + slope * np.log10(d))

plt.figure(figsize=(7, 4.2))
d = np.geomspace(D[0], 100_000, 50)
plt.plot(
    d,
    line(d),
    "--",
    color="#D55E00",
    label=f"line through the first three: $L \\propto D^{{{slope:.2f}}}$",
)
plt.plot(D[:3], L[:3], "o", color="k", label="fitted on")
plt.plot(D[3:], L[3:], "o", mfc="white", color="k", label="measured, not fitted on")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("training images D (log)")
plt.ylabel("test loss at the dot (log)")
plt.title("The line through the small sizes promises more than the large sizes give")
plt.legend(frameon=False)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(
    f"slope {slope:.3f}: 10x the images multiplies the loss by 10^{slope:.3f} = {10**slope:.2f}"
)
for n, l in zip(D[3:], L[3:]):
    print(f"{int(n):6,d} images: the line says {line(n):.3f}, measured {l:.3f}")

**Reading.** A line through 1,010 to 7,070 images has slope −0.23: 10× the images
multiplies the loss by 0.59. The line says 0.183 at 70,700 images; the run gives 0.225.
Both larger sizes lie above the line: the curve bends away from it. A slope holds inside
the range it was measured on, and a line drawn further is a guess, not a measurement.

## 3.2.4 Kaplan's three laws

Kaplan et al. (2020), *Scaling Laws for Neural Language Models*, measured the same form
over far wider ranges. A **language model** reads a text and predicts the next piece of it,
a **token** (a word or a part of one); the whole model is trained from scratch. Its
score is the **test loss**, the cross-entropy per token on held-out text, in nats (the
natural logarithm, as `nn.CrossEntropyLoss` uses). Three resources:

| law | resource | unit | slope $-\alpha$ | 10× buys | holds when |
|---|---|---|---|---|---|
| $L(N)$ | parameters $N$ | parameters | −0.076 | loss × 0.84 | the data is not the limit |
| $L(D)$ | dataset size $D$ | tokens | −0.095 | loss × 0.80 | the model is large enough, stopped at its best step |
| $L(C)$ | compute $C$ | PF-days | −0.050 | loss × 0.89 | every budget gets the model size that does best with it |

- $N$ leaves out the **embedding table**, the lookup table that turns each token into a
  vector before the model works on it; Kaplan counts only the parameters that compute.
- **Compute** is the arithmetic of training: every multiplication and addition is an
  operation, and training a model with $N$ parameters on $D$ tokens costs about
  $C \approx 6 \cdot N \cdot D$ operations: per parameter and per token about 2 for the
  forward pass and 4 for the backward pass. A **PF-day** is $10^{15}$ operations per second
  for a day, $8.64 \cdot 10^{19}$ operations.

10× the data multiplies the loss of a language model by 0.80, and of your Food-101 head, over
its first three sizes, by 0.59. The same form, another slope: other studies measured it in
translation, image classification and speech recognition (Hestness et al. 2017), and in
image and video generation (Henighan et al. 2020), each with its own slope. A slope belongs
to the problem it was measured on; the slope of your problem is measured on your problem.

## 3.2.5 Why the line bends: data and parameters grow together

In 3.2.3 only the data grew. The model stayed the same: a frozen encoder and a linear head.
Kaplan's equation 1.5 fits the loss of a model with $N$ parameters on $D$ tokens in one
formula:

$$L(N, D) = \left[\left(\frac{N_c}{N}\right)^{\alpha_N/\alpha_D} + \frac{D_c}{D}\right]^{\alpha_D}$$

with $\alpha_N = 0.076$, $\alpha_D = 0.103$, $N_c = 6.4 \cdot 10^{13}$ and
$D_c = 1.8 \cdot 10^{13}$; this joint fit gives a slightly different $\alpha_D$ than the
data law alone (0.095). Read it in two steps. When $D$ is small, the term $D_c / D$ is
large and the data sets the loss, whatever $N$ is. When $D$ grows without end, that term
goes to zero and what is left is the loss of the model itself: a floor set by $N$.

**Expect.** The next cell draws equation 1.5 against $D$ for three model sizes, 10× apart
(100× from the smallest to the largest).
On little data, does the larger model do better? Which line bends first?

In [ ]:
ND = {"aN": 0.076, "aD": 0.103, "Nc": 6.4e13, "Dc": 1.8e13}


def kaplan_nd(n, d):
    # Kaplan et al. (2020), eq. 1.5: test loss of a model with n parameters on d tokens
    return ((ND["Nc"] / n) ** (ND["aN"] / ND["aD"]) + ND["Dc"] / d) ** ND["aD"]


SIZES_N = [1e7, 1e8, 1e9]
tokens = np.geomspace(1e7, 1e12, 100)
plt.figure(figsize=(7, 4.2))
for n, c in zip(SIZES_N, COLOURS[2:]):
    plt.plot(tokens, kaplan_nd(n, tokens), color=c, label=f"N = {n:.0e} parameters")
    plt.axhline(kaplan_nd(n, np.inf), color=c, ls=":", lw=1)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("dataset size D (tokens, log)")
plt.ylabel("test loss, eq. 1.5 (nats per token, log)")
plt.title(
    "On little data the sizes tie; with N fixed, more data bends to a floor (dotted)"
)
plt.legend(frameon=False)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"{'D (tokens)':>11s}" + "".join(f"{f'N = {n:.0e}':>12s}" for n in SIZES_N))
for d in [1e7, 1e9, 1e11, np.inf]:
    print(f"{d:11.0e}" + "".join(f"{kaplan_nd(n, d):12.2f}" for n in SIZES_N))

**Reading.** At the left, on $10^{7}$ tokens, the three models lie on top of each other:
4.43, 4.41 and 4.41 nats per token for 100× more parameters. On little data, extra
parameters buy almost nothing; the data is the limit. That is the paper copied onto your
data: the $10^{9}$ model reaches its floor near $10^{11}$ tokens, and on $10^{7}$, a
ten-thousandth of that, it does no better than the small one. In equation 1.5, where every
model is stopped at its best step, a bigger model never does worse; it only stops paying.

To the right the lines part. The smallest model leaves the others between $10^{8}$ and
$10^{9}$ tokens and runs flat into its floor (3.29); the 100× larger model keeps falling to
about $10^{11}$. More data does not get a model below its own floor; only more parameters
move the floor down. So data and parameters grow together.

**Two ratios from one paper, for two questions.**

1. *How much data does a bigger model need so that the data does not hold it back?* With
   every model trained on the data until its best step, Kaplan's answer comes from
   equation 1.5. Multiply $N$ by 8 and the model's own term $(N_c/N)^{\alpha_N/\alpha_D}$
   shrinks by $8^{\alpha_N/\alpha_D}$. For the data term $D_c/D$ to keep the same share of
   the loss, it must shrink by the same factor, so $D$ grows by
   $8^{\alpha_N/\alpha_D} = 8^{0.74} = 4.6$: 8× the parameters with about 5× the data (the
   first of the two slides).
2. *With a fixed compute budget $C \approx 6 N D$, how to split it?* Here every model is
   stopped when the budget runs out, often well before its best step. Kaplan found the
   lowest loss for a budget with a large model stopped early: 8× the parameters with 2.2×
   the data. His runs lowered the learning rate over one planned length (a learning-rate
   schedule), and a short run was read partway through it, where it looks worse than it
   would at its own end. Hoffmann et al. (2022) lowered every run's learning rate to its
   own end, and then more data paid off more: 8× the parameters with 8× the data. Their
   Chinchilla (70 billion parameters on 1.4 trillion tokens,
   $6 \cdot N \cdot D \approx 5.9 \cdot 10^{23}$ operations) beat Gopher, sized along
   Kaplan's advice (280 billion on 300 billion tokens, $5.0 \cdot 10^{23}$), at about the
   same compute (the second slide). A ratio belongs to the study and the question that
   produced it.

Your own curve of 3.2.3 also held the model fixed: the frozen encoder with its linear head.
Its bend can come from the photos themselves (some dishes look alike, and some training
labels are wrong) or from the model; this notebook does not separate the two. The next
section gives you a parameter axis of your own.

## 3.2.6 A bottleneck whose width you choose

To see the parameter side, you need a model whose size limits it. A good example would train
a whole model on far more data: `dinov2-large` itself has about 300 million parameters and
learnt from 142 million images on a cluster of GPUs. That is beyond this lesson. And
its vectors are already laid out so well that a linear map separates the 101 dishes, so a
head on top has almost nothing left to learn.

So this section builds the limit in on purpose: a **bottleneck**, a hidden layer of $w$
units between the vector and the 101 dishes (`Linear(1024, w)`, GELU, `Linear(w, 101)`).
Everything the head knows about a photo has to pass through those $w$ numbers, so $w$ sets
how much the model can use. It is a stand-in for model size, not a good model.

**Question.** What does each doubling of the bottleneck buy?

**Setup.** Widths 16, 32, 64, 128 and 256, on the same five sizes, with the same rule as
3.2.2, one learning rate for all and no regularisation besides the choice of the step. The
linear head of 3.2.2 has no bottleneck: it passes all 1024 numbers on, so it is drawn as
the reference the widths approach. The 30 runs take about a minute, so they run in
`food_scaling.py` and store `results/food_scaling.json`; the next cell loads it.

**Expect.** Write down: with the data fixed, what does the step from 16 to 32 units buy,
and what does the step from 128 to 256 buy?

In [ ]:
# True: run the sweep of food_scaling.py again (about a minute) before loading it.
# Under notebooktester it runs in a test mode of seconds and stores its result in results/test.
RERUN = param(False, test=True)
RESULTS = Path("results", param("", test="test"))
if RERUN:
    subprocess.run(
        [sys.executable, "food_scaling.py", *param([], test=["--test"])], check=True
    )
sweep = json.loads((RESULTS / "food_scaling.json").read_text(encoding="utf-8"))

widths = sorted({r["width"] for r in sweep})
sizes = sorted({r["n"] for r in sweep})
plt.figure(figsize=(8, 4.4))
for w, c in zip(widths, ["k", *COLOURS]):
    rs = sorted([r for r in sweep if r["width"] == w], key=lambda r: r["n"])
    label = "linear, no bottleneck" if w == 0 else f"{w} units"
    style = dict(ls="--", lw=1.5) if w == 0 else dict(ls="-", lw=1.5)
    plt.plot(
        [r["n"] for r in rs],
        [r["held"] for r in rs],
        marker="o",
        color=c,
        label=label,
        **style,
    )
plt.xscale("log")
plt.yscale("log")
plt.xlabel("training images (log)")
plt.ylabel("test loss at the dot (log)")
plt.title("Each doubling of the bottleneck buys less than the one before")
plt.legend(frameon=False, fontsize=8, ncol=2)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"{'head':>10s}" + "".join(f"{n:>9,d}" for n in sizes))
for w in widths:
    rs = sorted([r for r in sweep if r["width"] == w], key=lambda r: r["n"])
    name = "linear" if w == 0 else f"{w} units"
    print(f"{name:>10s}" + "".join(f"{r['held']:9.3f}" for r in rs))

**Reading.** Read along a row of the table. At 20,200 images, 16 → 32 units takes 0.103 off
the loss, 32 → 64 0.049, 64 → 128 0.014 and 128 → 256 0.004. Every size up to 20,200 shows
the same: with the data fixed, each doubling takes less off than the one before, and the
widths flatten onto the linear head. It is the bend of 3.2.3 mirrored: there the model was
fixed and more data bought less and less; here the data is fixed and more parameters buy
less and less. (At 70,700 images 256 units lands
slightly above 128, with one learning rate for all; another seed could swap the two.)

The stand-in has its limit: the widths lie furthest apart on little data, where the sizes
of equation 1.5 tie. A bottleneck limits how much of a vector the encoder already learnt
passes on; it does not learn the photos itself, so that part of 3.2.5 it cannot show.

## 3.2.7 Beyond the top score

**Question.** The top score is the linear head on all 70,700 images: 0.935 test accuracy.
What does the curve say beyond that number?

**Reading, from the table of 3.2.2.** 20,200 images give 0.923: 3.5× fewer labelled photos
for 1.2 points of accuracy, and a best step of 468 instead of 1,104. Labelling is often the
most expensive part of a problem. Measured on parts of the labels you have, the curve shows
what more labels bought so far; drawn further, it is a guess (3.2.3). Whether 1.2 points are
worth 50,500 more labels is a choice for whoever pays for the labels; the curve gives the
price.

A smaller head would not make an answer cheaper: the encoder runs its 300 million
parameters for every photo, and every head here is small next to it. On this problem the
saving lies in labels and steps.

## What you now know

- A **scaling law** says the loss falls as a power of the data, the parameters or the
  compute: a straight line on log-log axes, whose slope says what 10× more buys. Kaplan's
  laws for language models have slopes −0.076, −0.095 and −0.050; your Food-101 head had
  −0.23 over its first three sizes. The slope belongs to the problem it was measured on.
- A line holds inside the range it was measured on. Drawn further, your own line promised
  0.183 at 70,700 images; the run gave 0.225.
- **Data and parameters grow together.** In Kaplan's equation 1.5, on little data, models
  100× apart in size give the same loss: the data is the limit. With the model fixed, more
  data bends towards the model's floor; only more parameters move it. How much more data a
  bigger model needs depends on the study and the question: 8× the parameters with 5×, 2.2×
  or 8× the data.
- More data also asks for more optimizer steps before the best test loss; a number of
  epochs from another dataset says little about yours.
- **A bottleneck as a stand-in for model size:** on the same data, each doubling of its
  width takes less off the loss than the one before. It shows the parameter side only in
  part, because it sits on a frozen encoder.
- **Beyond the top score:** 3.5× fewer labelled photos cost 1.2 points of accuracy; the
  curve gives the price of more labels, and the budget decides.

**When you do this for real.** A paper's model, size and training length come with the
paper's data. With a hundred or a thousand times less, measure your own curve: a few sizes
of the data, a few sizes of the model, each trained past its best step. Then choose the
model, the steps and the data from that curve, and dare to start small.

## Exercises

**E1: a floor in the fit.** Fit $L(D) = L_\infty + a \cdot D^{-\alpha}$ to the five points of
3.2.3 instead of a straight line (`scipy.optimize.curve_fit` fits a formula with free
constants to points). How large is the floor $L_\infty$? Extrapolate both fits to 700,000
images: which would you believe, and why can neither be checked on this data?

**E2: your own slope on another dataset.** Repeat 3.2.2 and 3.2.3 on EuroSAT
(`VectorCache.from_hub("pttrn-io/eurosat-dinov2-small")`, satellite tiles in 10 classes).
Predict first: is its slope steeper or flatter than Food-101's? Both the dataset and the
encoder differ, so which of the two could a different slope come from?